# Notebook 03 — Compare **all nine** fault diagnosis classifiers

This notebook uses the **same functions as `main.py`**. It shows how the
seven classical methods and the two deep-learning methods are trained
and compared.

**Run `main.py` in TRAIN mode for the complete two-stage system.**
This notebook focuses on *diagnosis-only* results, which use known faulty
windows. The full-system results also include the autoencoder's missed
faults and false alarms.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT = Path.cwd()
if PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent
sys.path.insert(0, str(PROJECT))

from data import load_windows
DATA_FILE = PROJECT / "data" / "PS1_demo.csv"
DATA_FORMAT = "cycles"

from sklearn.model_selection import train_test_split
from faults import make_faults, FAULT_NAMES
from diagnosis import (
    CLASSIFIER_NAMES, train_all_classifiers, diagnose_faults, make_metrics
)

## 1. Keep original cycles apart BEFORE fault injection

In [ ]:
windows, cycle_ids = load_windows(DATA_FILE, DATA_FORMAT, windows_per_cycle=3)

train_cycles, remaining_cycles = train_test_split(
    np.unique(cycle_ids), test_size=0.30, random_state=42
)
validation_cycles, test_cycles = train_test_split(
    remaining_cycles, test_size=0.50, random_state=42
)
healthy_train = windows[np.isin(cycle_ids, train_cycles)]
healthy_val = windows[np.isin(cycle_ids, validation_cycles)]
healthy_test = windows[np.isin(cycle_ids, test_cycles)]
print("Healthy train / validation / test:",
      len(healthy_train), len(healthy_val), len(healthy_test))

## 2. Generate gain, bias, constant, and constant-zero faults

In [ ]:
X_train, y_train = make_faults(healthy_train)
X_val, y_val = make_faults(healthy_val)
X_test, y_test = make_faults(healthy_test)

print("Training windows:", X_train.shape)
print("Fault types:", FAULT_NAMES)

## 3. Train all nine diagnosis classifiers

In [ ]:
# To experiment with a shorter run, you can temporarily change EPOCHS = 5.
EPOCHS = 12

models, deep_scaler, histories = train_all_classifiers(
    X_train, y_train,
    X_val, y_val,
    epochs=EPOCHS, selected_names=CLASSIFIER_NAMES
)
print("Trained:", list(models))

## 4. Evaluate every method on exactly the same unseen TEST cycles

We compare precision, recall, F1 and accuracy for fault diagnosis.
These are **not** the complete two-stage system metrics.

In [ ]:
rows = []
predictions = {}

for name, model in models.items():
    pred = diagnose_faults(model, X_test, name, deep_scaler)
    predictions[name] = pred
    rows.append({
        "classifier": name,
        **make_metrics(y_test, pred, labels=[1, 2, 3, 4])
    })

scores = pd.DataFrame(rows)
print(scores.to_string(index=False))

## 5. Compare all methods on one graph

In [ ]:
ax = scores.set_index("classifier")[
    ["precision", "recall", "f1", "accuracy"]
].plot.bar(figsize=(12, 5), ylim=(0, 1))

ax.set_title("Fault diagnosis — same test windows for all classifiers")
ax.set_ylabel("Test metric")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 6. Show a confusion matrix for EACH classifier

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

for name in CLASSIFIER_NAMES:
    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay.from_predictions(
        y_test, predictions[name],
        labels=[1, 2, 3, 4],
        display_labels=[FAULT_NAMES[i] for i in [1, 2, 3, 4]],
        ax=ax, xticks_rotation=30, values_format="d",
    )
    ax.set_title(name)
    plt.tight_layout()
    plt.show()

## 7. Investigate any method in more detail

In the two deep-learning models, we can also inspect train and
validation loss. The same plots are saved automatically by `main.py`.

In [ ]:
for name, history in histories.items():
    plt.figure(figsize=(7, 4))
    plt.plot(history["loss"], label="training loss")
    plt.plot(history["val_loss"], label="validation loss")
    plt.title(name)
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.show()

### Where is the learned autoencoder threshold?

Notebook **02** shows how threshold candidates are compared on *validation*
data. `main.py` then saves `saved_models/detector_settings.json` and uses
that threshold on unseen test examples. The final test predictions,
diagnosis-only comparison, and **complete detector + classifier comparison**
are saved in `results/`.

Do **not** pick a classifier from the final TEST scores and report that
same test score as a fresh, unbiased evaluation of the chosen model.